# `transform_kpis.ipynb` - Cálculo de indicadores (Densidad, normalización, IOI)

Implementa las tres formulaciones matemáticas exigidas por la especificación (sección 4):

1. **Densidad Energética por Habitante**: `Densidad = (Potencia Total / Población) × 10.000` - habitantes/10.000 hab. Rango esperado `[0,00, +∞)`, valores típicos entre 0 y 500.
2. **Normalización Min-Max**: `f(x) = (x - xmin) / (xmax - xmin)` - aplicada tanto a la renta como a la densidad, sobre el conjunto de los 11 distritos. Rango `[0,00, 1,00]`.
3. **Índice de Oportunidad de Inversión (IOI)**: `IOI = (Renta_norm × (1 - Densidad_norm)) × 100` - combina poder adquisitivo alto con déficit de oferta energética. Rango `[0,00, 100,00]`.

> Depende de `config.ipynb` (usa `FACTOR_DENSIDAD`) y de los DataFrames producidos por `extract_distritos.ipynb`, `extract_socioeconomico.ipynb` y `transform_geoprocessing.ipynb`.

In [ ]:
import logging

import pandas as pd
import geopandas as gpd

## 1. Unir geometría + socioeconomía + potencia agregada

In [ ]:
def construir_tabla_maestra(gdf_distritos: gpd.GeoDataFrame, df_socio: pd.DataFrame, df_potencia: pd.DataFrame) -> gpd.GeoDataFrame:
    """Cruza geometria de distritos + tabla socioeconomica + potencia agregada, por nombre_distrito."""
    logger.info("Uniendo geometria, tabla socioeconomica y potencia agregada por distrito...")
    master = gdf_distritos.merge(df_socio, on="nombre_distrito", how="left")
    master = master.merge(df_potencia, on="nombre_distrito", how="left")

    faltantes = master[master["poblacion_total"].isna()]["nombre_distrito"].tolist()
    if faltantes:
        logger.warning(f"Distritos sin dato socioeconomico tras el cruce: {faltantes}")

    logger.info(f"Tabla maestra construida: {len(master)} distritos.")
    return master

## 2. Densidad Energética por Habitante

In [ ]:
def calcular_densidad_energetica(master: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Densidad = (Potencia Total kW / Poblacion) * FACTOR_DENSIDAD (habitantes/10.000 hab)."""
    master = master.copy()
    master["densidad_energetica"] = (
        master["potencia_total_kw"] / master["poblacion_total"]
    ) * FACTOR_DENSIDAD
    logger.info(
        f"Densidad energetica calculada. Rango: "
        f"[{master['densidad_energetica'].min():.2f}, {master['densidad_energetica'].max():.2f}] kW/10k hab."
    )
    return master

## 3. Normalización Min-Max

In [ ]:
def normalizar_minmax(serie: pd.Series) -> pd.Series:
    """f(x) = (x - xmin) / (xmax - xmin), sobre el conjunto de los 11 distritos."""
    xmin, xmax = serie.min(), serie.max()
    if xmax == xmin:
        logger.warning("Rango nulo (xmax == xmin) al normalizar; se devuelve una serie de ceros.")
        return serie * 0.0
    return (serie - xmin) / (xmax - xmin)

## 4. Índice de Oportunidad de Inversión (IOI)

In [ ]:
def calcular_ioi(master: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """IOI = (Renta_norm * (1 - Densidad_norm)) * 100."""
    master = master.copy()
    master["renta_norm"] = normalizar_minmax(master["renta_media_neta_persona"])
    master["densidad_norm"] = normalizar_minmax(master["densidad_energetica"])
    master["IOI"] = (master["renta_norm"] * (1 - master["densidad_norm"])) * 100
    logger.info(f"IOI calculado. Rango: [{master['IOI'].min():.2f}, {master['IOI'].max():.2f}]")
    return master

## Prueba rápida

Con la tabla socioeconómica real y una potencia agregada sintética de ejemplo (sin depender de la API ni del spatial join), para comprobar que las tres fórmulas dan el rango esperado.

In [ ]:
import random
random.seed(1)

gdf_distritos_prueba3 = extraer_distritos()
df_socio_prueba2 = extraer_tabla_socioeconomica()
df_potencia_prueba = pd.DataFrame({
    "nombre_distrito": sorted(POBLACION_DISTRITOS.keys()),
    "potencia_total_kw": [random.choice([0, 22, 44, 66, 88, 110]) for _ in range(11)],
    "num_estaciones": [random.randint(0, 4) for _ in range(11)],
})

master_prueba = construir_tabla_maestra(gdf_distritos_prueba3, df_socio_prueba2, df_potencia_prueba)
master_prueba = calcular_densidad_energetica(master_prueba)
master_prueba = calcular_ioi(master_prueba)

cols = ["nombre_distrito", "poblacion_total", "renta_media_neta_persona",
        "potencia_total_kw", "densidad_energetica", "renta_norm", "densidad_norm", "IOI"]
print(master_prueba[cols].sort_values("IOI", ascending=False).to_string(index=False))

---
✅ **KPIs verificados**: densidad ≥ 0, normalizaciones en `[0,1]`, IOI en `[0,100]`.